# Logistic Regression Classification Masterclass

End-to-end binary classification using Bank Marketing-style data, Statsmodels, scikit-learn, imbalance handling, calibration, diagnostics, and business thresholding.

## 1. Logistic model

$P(Y=1|X)=\sigma(\beta_0+X\beta)$, where $\sigma(z)=1/(1+e^{-z})$. A coefficient $\beta_j$ changes log-odds; $e^{\beta_j}$ is the adjusted odds multiplier.

In [ ]:
from pathlib import Path
import json, joblib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import *
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore'); SEED=43; np.random.seed(SEED)
ROOT=Path.cwd(); DATA=ROOT/'data'/'bank_marketing_teaching.csv'
if not DATA.exists():
    exec((ROOT/'scripts'/'generate_teaching_data.py').read_text(), {'__name__':'__main__','__file__':str(ROOT/'scripts'/'generate_teaching_data.py')})
df=pd.read_csv(DATA); df['target']=(df['y']=='yes').astype(int)
print(df.shape, df['target'].mean())

## 2. Data audit and leakage

`duration` is known only after the current call ends. It may explain historical outcomes, but it must be excluded from a pre-call targeting model.

In [ ]:
audit=pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'unique':df.nunique(dropna=False)})
print('duplicates:',df.duplicated().sum()); display(audit)
df=df.drop_duplicates().reset_index(drop=True)

## 3. Univariate, bivariate, and multivariate EDA

In [ ]:
num=['age','balance','day','duration','campaign','pdays','previous']
cat=['job','marital','education','default','housing','loan','contact','month','poutcome']
display(df[num].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax,c in zip(axes.ravel(),['age','balance','campaign','duration']):
    for y,label in [(0,'No'),(1,'Yes')]: ax.hist(df.loc[df.target==y,c],bins=35,alpha=.5,density=True,label=label)
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
assoc=[]
for c in num:
    r,p=pointbiserialr(df.target,df[c]); assoc.append((c,r,p))
display(pd.DataFrame(assoc,columns=['feature','point_biserial_r','p_value']).sort_values('point_biserial_r',key=abs,ascending=False))
for c in ['poutcome','contact','month','job']:
    display(df.groupby(c,observed=True).target.agg(['mean','count']).sort_values('mean',ascending=False))

## 4. Leakage-safe split and preprocessing

In [ ]:
features=[c for c in df.columns if c not in ['y','target','duration']]
X=df[features]; y=df.target
Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=.4,stratify=y,random_state=SEED)
Xv,Xte,yv,yte=train_test_split(Xtmp,ytmp,test_size=.5,stratify=ytmp,random_state=SEED)
num=[c for c in X.columns if X[c].dtype!='object']; cat=[c for c in X.columns if X[c].dtype=='object']
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',drop='first'))]),cat)])
print(pd.DataFrame({'rows':[len(Xtr),len(Xv),len(Xte)],'rate':[ytr.mean(),yv.mean(),yte.mean()]},index=['train','validation','test']))

## 5. Baseline and logistic regression

In [ ]:
def metrics(y,p,t=.5):
    z=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,z).ravel()
    return {'threshold':t,'roc_auc':roc_auc_score(y,p),'avg_precision':average_precision_score(y,p),'precision':precision_score(y,z,zero_division=0),'recall':recall_score(y,z),'specificity':tn/(tn+fp),'f1':f1_score(y,z),'mcc':matthews_corrcoef(y,z),'log_loss':log_loss(y,p),'brier':brier_score_loss(y,p)}
dummy=Pipeline([('pre',pre),('model',DummyClassifier(strategy='prior'))]).fit(Xtr,ytr)
base=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=4000,solver='liblinear',random_state=SEED))]).fit(Xtr,ytr)
display(pd.DataFrame([metrics(yv,dummy.predict_proba(Xv)[:,1]),metrics(yv,base.predict_proba(Xv)[:,1])],index=['dummy','logistic']))

## 6. ROC, precision-recall, calibration, and confusion matrix

In [ ]:
p=base.predict_proba(Xv)[:,1]
fpr,tpr,_=roc_curve(yv,p); pr,rc,_=precision_recall_curve(yv,p); obs,pred=calibration_curve(yv,p,n_bins=10,strategy='quantile')
fig,ax=plt.subplots(1,3,figsize=(16,4)); ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],'--'); ax[0].set_title('ROC')
ax[1].plot(rc,pr); ax[1].axhline(yv.mean(),ls='--'); ax[1].set_title('Precision-Recall')
ax[2].plot(pred,obs,'o-'); ax[2].plot([0,1],[0,1],'--'); ax[2].set_title('Calibration'); plt.tight_layout(); plt.show()
ConfusionMatrixDisplay.from_predictions(yv,p>=.5); plt.show()

## 7. Regularization and cross-validation

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
grid=GridSearchCV(Pipeline([('pre',pre),('model',LogisticRegression(max_iter=4000,solver='liblinear',random_state=SEED))]),{'model__penalty':['l1','l2'],'model__C':[.03,.1,.3,1,3],'model__class_weight':[None,'balanced']},scoring={'ap':'average_precision','auc':'roc_auc'},refit='ap',cv=cv,n_jobs=1)
grid.fit(Xtr,ytr); print(grid.best_params_,grid.best_score_)
best=grid.best_estimator_; pv=best.predict_proba(Xv)[:,1]

## 8. Imbalance handling: class weighting versus SMOTE

In [ ]:
smote=ImbPipeline([('pre',pre),('smote',SMOTE(random_state=SEED)),('model',LogisticRegression(max_iter=4000,solver='liblinear',C=.3,random_state=SEED))]).fit(Xtr,ytr)
display(pd.DataFrame([metrics(yv,pv),metrics(yv,smote.predict_proba(Xv)[:,1])],index=['tuned','smote']))

## 9. Business threshold optimization

Choose the threshold on validation data only. Here a false negative costs 6 units and a false positive costs 1 unit.

In [ ]:
rows=[]
for t in np.linspace(.01,.8,160):
    z=(pv>=t).astype(int); tn,fp,fn,tp=confusion_matrix(yv,z).ravel(); rows.append((t,fp+6*fn,precision_score(yv,z,zero_division=0),recall_score(yv,z)))
th=pd.DataFrame(rows,columns=['threshold','cost','precision','recall']); threshold=float(th.loc[th.cost.idxmin(),'threshold'])
print('selected threshold:',round(threshold,3)); th.plot(x='threshold',y=['precision','recall'],figsize=(8,4)); plt.show()

## 10. Statsmodels inference and adjusted odds ratios

In [ ]:
d=Xtr.copy(); d['target']=ytr.to_numpy()
formula='target ~ age + balance + campaign + previous + C(housing) + C(loan) + C(contact) + C(poutcome) + C(education) + C(month)'
glm=smf.glm(formula,d,family=sm.families.Binomial()).fit(); print(glm.summary())
ci=glm.conf_int(); odds=pd.DataFrame({'coef':glm.params,'odds_ratio':np.exp(glm.params),'ci_low':np.exp(ci[0]),'ci_high':np.exp(ci[1]),'p_value':glm.pvalues})
display(odds.sort_values('p_value').head(20))

## 11. Diagnostics

Inspect VIF/collinearity, Pearson residuals, leverage, Cook distance, separation, linearity in the logit, and binned observed-versus-fitted calibration.

In [ ]:
infl=glm.get_influence(observed=False).summary_frame(); resid=glm.resid_pearson
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].scatter(glm.fittedvalues,resid,s=12,alpha=.3); ax[0].axhline(0,ls='--'); ax[0].set_title('Pearson residuals')
ax[1].scatter(infl.hat_diag,infl.standard_resid,s=12,alpha=.3); ax[1].axhline(3,ls=':'); ax[1].axhline(-3,ls=':'); ax[1].set_title('Leverage and residuals'); plt.tight_layout(); plt.show()
print(infl.nlargest(10,'cooks_d')[['cooks_d','hat_diag','standard_resid']])

## 12. Final refit, untouched test evaluation, and serialization

In [ ]:
Xtv=pd.concat([Xtr,Xv]); ytv=pd.concat([ytr,yv]); final=best.fit(Xtv,ytv); pte=final.predict_proba(Xte)[:,1]
result=pd.DataFrame([metrics(yte,pte,.5),metrics(yte,pte,threshold)],index=['threshold_0.5','business_threshold']); display(result)
art=ROOT/'artifacts'; art.mkdir(exist_ok=True); joblib.dump(final,art/'final_logistic_pipeline.joblib')
(art/'selected_threshold.json').write_text(json.dumps({'threshold':threshold,'false_positive_cost':1,'false_negative_cost':6},indent=2)); result.to_json(art/'model_metrics.json',orient='index',indent=2)

## 13. Model card

**Intended use:** rank prospective customers before campaign contact. **Excluded leakage:** current-call duration. **Do not infer causality** from coefficients. Monitor prevalence, category drift, calibration, average precision, subgroup recall, contact capacity, and realized campaign value. Revalidate the threshold whenever costs or capacity change.